In [1]:
import  os

%pwd

'C:\\Users\\Lenovo\\image classification\\image-classification-\\research'

In [2]:
os.chdir("../")

In [3]:
%pwd

'C:\\Users\\Lenovo\\image classification\\image-classification-'

In [4]:
import  tensorflow as tf

In [5]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [20]:
from  dataclasses import  dataclass
from pathlib import  Path

In [33]:
@dataclass(frozen=True)
class EvaluationConfig :
    path_of_model : Path
    training_data : Path
    params_image_size : list
    params_batch_size : int
    all_params : dict

In [34]:
from cnnclassifier.constants import *
from cnnclassifier.utils.common import read_yaml , create_directories , save_json

In [39]:
class ConfigurationManager :
    def __init__(
    self ,
    config_filepath = Config_File_path  ,
    params_filepath = Paramas_File_path) :

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root] )

    def get_validation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
        path_of_model = "artifacts/training/model.h5" ,
        training_data = "artifacts/data_ingestion/train" ,
        all_params = self.params ,
        params_batch_size = self.params.BATCH_SIZE ,
        params_image_size = self.params.IMAGE_SIZE ,
        )
        return eval_config


In [36]:
from urllib.parse import urlparse

In [37]:
class Evaluation :
    def __init__ (self,  config : EvaluationConfig) :
        self.config = config

    def valid_generator  (self) :
        datagenerator_kwargs = dict (
            rescale =  1/.255 ,
            validation_split = 0.3 ,
        )
        dataflow_kwargs = dict (
            target_size = self.config.params_image_size[:-1] ,
            batch_size = self.config.params_batch_size ,
            interpolation = "bilinear",
        )
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(**datagenerator_kwargs)

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory = self.config.training_data ,
            subset = "validation",
            shuffle = False,
            **dataflow_kwargs

        )
    @staticmethod
    def load_model(path : Path) -> tf.keras.Model :
        return tf.keras.models.load_model(path)

    def evaluation (self) :
        self.model = self.load_model(self.config.path_of_model)
        self.valid_generator()
        self.score = model.evaluate(self.valid_generator)

    def save_score (self) :
        scores = {"loss " : self.score[0] , "accuracy" : self.score[1] }
        save_json(path = Path("scores.json") , data = scores)



In [40]:
try :
    config = ConfigurationManager()
    val_config = config.get_validation_config()
    evaluation = Evaluation(val_config)
    evaluation.evaluation()
    evaluation.save_score()
except Exception as e :
    raise e

Found 116 images belonging to 2 classes.
8/8 ━━━━━━━━━━━━━━━━━━━━ 7s 760ms/step - accuracy: 0.6293 - loss: 49.9874
